# Activity 1: RAGAS Evaluation — Fireworks vs OpenAI

Compares the RAG pipeline powered by Fireworks `gpt-oss-20b` against OpenAI `gpt-4.1-mini`.

**Metrics (all 5):**
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Answer Relevancy** — does the answer actually address the question?
- **Context Precision** — are the retrieved chunks relevant to the question?
- **Context Recall** — did retrieval capture all the important information?
- **Answer Correctness** — is the answer factually correct against the reference?

Ground truth reference answers are generated by passing the source PDF text directly to the LLM — grounded in the actual document, not model memory.

Both pipelines are tagged in LangSmith so token usage and cost can be compared per provider.

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "session-10-rag-eval")
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]

print("LangSmith project:", os.environ.get("LANGCHAIN_PROJECT"))
print("Fireworks chat model:", os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"))
print("Fireworks embedding model:", os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"))

LangSmith project: session-10-rag-eval
Fireworks chat model: accounts/fireworks/models/gpt-oss-20b
Fireworks embedding model: accounts/fireworks/models/qwen3-embedding-8b


## Load and Chunk Documents

In [15]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig

FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"
DATA_DIR = "data"

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("human", "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
     "Use the provided context to answer the query. "
     "Only use the provided context. If the answer is not in the context, say \"I don't know\".")
])


def tiktoken_len(text):
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


def load_and_chunk():
    loader = DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=750, chunk_overlap=0, length_function=tiktoken_len
    )
    return docs, splitter.split_documents(docs)


print("Loading documents...")
raw_docs, chunks = load_and_chunk()
full_doc_text = "\n\n".join(doc.page_content for doc in raw_docs)

print(f"Loaded {len(raw_docs)} pages → {len(chunks)} chunks")
print(f"Total document length: {tiktoken_len(full_doc_text)} tokens")

Loading documents...
Loaded 22 pages → 42 chunks
Total document length: 24244 tokens


## Generate Ground Truth Reference Answers

Pass the full document text to the LLM to generate reference answers grounded in the actual PDF.

In [16]:
TEST_QUESTIONS = [
    "What vaccinations are recommended for kittens?",
    "At what age should a kitten be spayed or neutered?",
    "What are the signs of dental disease in cats?",
    "How often should an adult cat visit the vet?",
    "What parasites should cats be tested for regularly?",
    "What nutritional needs do senior cats have?",
    "What are the symptoms of feline hyperthyroidism?",
    "How should a cat's weight be monitored and managed?",
]

GROUND_TRUTH_PROMPT = ChatPromptTemplate.from_messages([
    ("human", "You are a veterinary expert. Using ONLY the document below, answer the question accurately and completely.\n\n"
     "DOCUMENT:\n{document}\n\nQUESTION:\n{question}\n\n"
     "If the document does not contain enough information to answer, say \"Not covered in document\".")
])

reference_llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"])
reference_chain = GROUND_TRUTH_PROMPT | reference_llm | StrOutputParser()

print("Generating ground truth reference answers from document...")
ground_truth = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    answer = reference_chain.invoke(
        {"document": full_doc_text, "question": q},
        config=RunnableConfig(tags=["ground-truth"], metadata={"provider": "ground-truth"})
    )
    ground_truth.append(answer)
    print(f"  [{i}/{len(TEST_QUESTIONS)}] done")

print("\nSample reference answer:")
print(ground_truth[0][:300])

Generating ground truth reference answers from document...
  [1/8] done
  [2/8] done
  [3/8] done
  [4/8] done
  [5/8] done
  [6/8] done
  [7/8] done
  [8/8] done

Sample reference answer:
The 2021 AAHA/AAFP Feline Life Stage Guidelines recommend the following vaccinations for kittens:

- Core vaccines: rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV).
- Feline leukemia virus (FeLV) vaccination is considered core for kitte


## Build RAG Pipelines

Each pipeline run is tagged with its provider so LangSmith can separate token usage and cost.

In [17]:
def build_pipeline(embedding_model, chat_model, chunks, provider_tag):
    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embedding_model,
        location=":memory:",
        collection_name="eval_collection",
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    chain = RAG_PROMPT | chat_model | StrOutputParser()

    # Tag every chain invocation with the provider so LangSmith can split them
    run_config = RunnableConfig(
        tags=[provider_tag],
        metadata={"provider": provider_tag}
    )

    def run(question):
        retrieved = retriever.invoke(question)
        context_text = "\n\n".join(doc.page_content for doc in retrieved)
        answer = chain.invoke({"query": question, "context": context_text}, config=run_config)
        contexts = [doc.page_content for doc in retrieved]
        return answer, contexts

    return run


print("Building Fireworks pipeline...")
fireworks_run = build_pipeline(
    embedding_model=OpenAIEmbeddings(
        model=os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"),
        openai_api_key=os.environ["FIREWORKS_API_KEY"],
        openai_api_base=FIREWORKS_BASE_URL,
        check_embedding_ctx_length=False,
        dimensions=4096,
    ),
    chat_model=ChatOpenAI(
        model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
        openai_api_key=os.environ["FIREWORKS_API_KEY"],
        openai_api_base=FIREWORKS_BASE_URL,
    ),
    chunks=chunks,
    provider_tag="fireworks-rag",
)

print("Building OpenAI pipeline...")
openai_run = build_pipeline(
    embedding_model=OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.environ["OPENAI_API_KEY"],
    ),
    chat_model=ChatOpenAI(
        model="gpt-4.1-mini",
        openai_api_key=os.environ["OPENAI_API_KEY"],
    ),
    chunks=chunks,
    provider_tag="openai-rag",
)

print("Both pipelines ready.")

Building Fireworks pipeline...
Building OpenAI pipeline...
Both pipelines ready.


## Run Both Pipelines

In [18]:
fireworks_results = []
openai_results = []

for i, (q, ref) in enumerate(zip(TEST_QUESTIONS, ground_truth), 1):
    print(f"[{i}/{len(TEST_QUESTIONS)}] {q}")
    fw_answer, fw_contexts = fireworks_run(q)
    oa_answer, oa_contexts = openai_run(q)
    fireworks_results.append({"question": q, "answer": fw_answer, "contexts": fw_contexts, "reference": ref})
    openai_results.append({"question": q, "answer": oa_answer, "contexts": oa_contexts, "reference": ref})

print("\nAll queries complete.")

[1/8] What vaccinations are recommended for kittens?
[2/8] At what age should a kitten be spayed or neutered?
[3/8] What are the signs of dental disease in cats?
[4/8] How often should an adult cat visit the vet?
[5/8] What parasites should cats be tested for regularly?
[6/8] What nutritional needs do senior cats have?
[7/8] What are the symptoms of feline hyperthyroidism?
[8/8] How should a cat's weight be monitored and managed?

All queries complete.


## RAGAS Evaluation

In [19]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, AnswerCorrectness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm),
    AnswerCorrectness(llm=evaluator_llm),
]


def build_ragas_dataset(results):
    return EvaluationDataset(samples=[
        SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=r["contexts"],
            reference=r["reference"],
        )
        for r in results
    ])


print("Evaluating Fireworks pipeline...")
fw_scores = evaluate(build_ragas_dataset(fireworks_results), metrics=metrics)

print("Evaluating OpenAI pipeline...")
oa_scores = evaluate(build_ragas_dataset(openai_results), metrics=metrics)

print("Done.")

/var/folders/6b/2bnrv4k52z53kz5qmw40mf6r0000gp/T/ipykernel_69910/1412945614.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, AnswerCorrectness
/var/folders/6b/2bnrv4k52z53kz5qmw40mf6r0000gp/T/ipykernel_69910/1412945614.py:3: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, AnswerCorrectness
/var/folders/6b/2bnrv4k52z53kz5qmw40mf6r0000gp/T/ipykernel_69910/1412945614.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be remo

Evaluating Fireworks pipeline...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating OpenAI pipeline...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Done.


## RAGAS Results

In [20]:
import pandas as pd

fw_df = fw_scores.to_pandas()
oa_df = oa_scores.to_pandas()

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

summary = pd.DataFrame({
    "Metric": metric_cols,
    "Fireworks gpt-oss-20b": [fw_df[m].mean().round(3) for m in metric_cols],
    "OpenAI gpt-4.1-mini": [oa_df[m].mean().round(3) for m in metric_cols],
})

print(summary.to_string(index=False))

            Metric  Fireworks gpt-oss-20b  OpenAI gpt-4.1-mini
      faithfulness                  0.434                0.731
  answer_relevancy                  0.542                0.661
 context_precision                  0.781                0.823
    context_recall                  0.923                0.940
answer_correctness                  0.566                0.544


In [21]:
print("=== Fireworks gpt-oss-20b ===")
display(fw_df[["user_input"] + metric_cols].round(3))

print("\n=== OpenAI gpt-4.1-mini ===")
display(oa_df[["user_input"] + metric_cols].round(3))

=== Fireworks gpt-oss-20b ===


,user_input,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
0,What vaccinations are recommended for kittens?,0.750,0.788,1.000,1.000,0.855
1,At what age should a kitten be spayed or neute...,0.000,0.000,0.833,0.667,0.550
2,What are the signs of dental disease in cats?,0.000,0.000,0.500,1.000,0.172
3,How often should an adult cat visit the vet?,0.800,0.874,1.000,1.000,0.571
4,What parasites should cats be tested for regul...,0.500,0.879,1.000,0.900,0.712
5,What nutritional needs do senior cats have?,0.806,0.863,0.917,0.900,0.781
6,What are the symptoms of feline hyperthyroidism?,0.000,0.000,0.000,1.000,0.196
7,How should a cat's weight be monitored and man...,0.619,0.933,1.000,0.920,0.689



=== OpenAI gpt-4.1-mini ===


,user_input,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
0,What vaccinations are recommended for kittens?,1.000,0.890,1.000,1.000,0.629
1,At what age should a kitten be spayed or neute...,0.000,0.000,0.833,0.667,0.475
2,What are the signs of dental disease in cats?,1.000,0.887,1.000,0.933,0.614
3,How often should an adult cat visit the vet?,1.000,0.892,0.750,1.000,0.404
4,What parasites should cats be tested for regul...,0.846,0.704,1.000,1.000,0.539
5,What nutritional needs do senior cats have?,1.000,0.912,1.000,1.000,0.875
6,What are the symptoms of feline hyperthyroidism?,0.000,0.000,0.000,1.000,0.196
7,How should a cat's weight be monitored and man...,1.000,1.000,1.000,0.923,0.623


## LangSmith Cost Analysis

Pulls token usage and cost for each provider's tagged runs directly from LangSmith.

In [22]:
from langsmith import Client

ls_client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
project = os.environ.get("LANGCHAIN_PROJECT", "session-10-rag-eval")


def get_cost_summary(tag):
    runs = list(ls_client.list_runs(
        project_name=project,
        filter=f'has(tags, "{tag}")',
        run_type="llm",
    ))
    total_prompt = sum(r.prompt_tokens or 0 for r in runs)
    total_completion = sum(r.completion_tokens or 0 for r in runs)
    total_cost = sum(r.total_cost or 0 for r in runs)
    return {
        "runs": len(runs),
        "prompt_tokens": total_prompt,
        "completion_tokens": total_completion,
        "total_tokens": total_prompt + total_completion,
        "total_cost_usd": round(total_cost, 6),
    }


print("Fetching LangSmith run data...")
fw_cost = get_cost_summary("fireworks-rag")
oa_cost = get_cost_summary("openai-rag")

cost_df = pd.DataFrame({
    "Provider": ["Fireworks gpt-oss-20b", "OpenAI gpt-4.1-mini"],
    "LLM Runs": [fw_cost["runs"], oa_cost["runs"]],
    "Prompt Tokens": [fw_cost["prompt_tokens"], oa_cost["prompt_tokens"]],
    "Completion Tokens": [fw_cost["completion_tokens"], oa_cost["completion_tokens"]],
    "Total Tokens": [fw_cost["total_tokens"], oa_cost["total_tokens"]],
    "Cost (USD)": [fw_cost["total_cost_usd"], oa_cost["total_cost_usd"]],
})

display(cost_df)

print("\nNote: Fireworks cost may show $0 in LangSmith (third-party provider).")
print("Check your Fireworks dashboard at fireworks.ai for accurate billing.")

Fetching LangSmith run data...


,Provider,LLM Runs,Prompt Tokens,Completion Tokens,Total Tokens,Cost (USD)
0,Fireworks gpt-oss-20b,8,21383,4272,25655,0
1,OpenAI gpt-4.1-mini,8,21266,1187,22453,0.005798



Note: Fireworks cost may show $0 in LangSmith (third-party provider).
Check your Fireworks dashboard at fireworks.ai for accurate billing.
